# Mission 0: Does it work? 🔥

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/00_ignite.ipynb)

> **From your syllabus:** warm-up. You will meet the digits dataset used in Missions 2 and 3 and see a trained model read handwriting.

Click a cell, press **Shift+Enter** to run it. That is the only skill you need right now.

> 📊 **Progress tracking: please read.** The next cell draws your progress map and records which exercises you finish. It sends your **name, roll number, exercise ID and time** to the instructor's progress sheet for this workshop, nothing else (no files, nothing from your Google account). Its full code is in the cell, so you can read it. Prefer not to be tracked? Type `anonymous` as your name, or set `TRACKING = False`; the map still works for you.
>
> Colab may ask *"Do you trust this code?"* because this notebook comes from GitHub (`anshid/MathMeetsML`, open source, MIT licence). That is expected: choose **Run anyway**.

In [ ]:
#@title Progress map (run me first; the code is short, read it)
# Records which exercises you finish so you can see your progress map and the instructor can see how the class is doing.
# Sent: your name, roll number, exercise id and time. Nothing else. Type "anonymous" as your name (or set TRACKING = False) to send nothing.
TRACKING = True
MISSION = 0

import json, os, time, base64, urllib.parse, urllib.request
from IPython.display import HTML, display, update_display

CLOUD_URL = "https://script.google.com/macros/s/AKfycbxekbOkIE5oRVA319mhEENS1phVLt7gPfRUernniP8kMopgIqUNHNN37xjF_3XZEuQWwQ/exec"
TOKEN = "3F5y-xKcPbZZPT49"
LAN_URL = os.environ.get("MML_LAN", "")
MISSIONS = {"0": {"title": "Ignite", "checkpoints": ["m00-ready", "m00-done"]}, "1": {"title": "Golf", "checkpoints": ["m01-grad", "m01-done"]}, "2": {"title": "Digit Reader", "checkpoints": ["m02-grad", "m02-done"]}, "3": {"title": "Selfie", "checkpoints": ["m03-ey", "m03-pca", "m03-done"]}, "4": {"title": "Mini-Google", "checkpoints": ["m04-markov", "m04-done"]}, "5": {"title": "Tiny AI", "checkpoints": ["m05-backprop", "m05-done"]}}
STORE = "progress.json"
SINKS = [u for u in (CLOUD_URL, LAN_URL) if u]


def _load():
    try:
        with open(STORE) as f:
            return json.load(f)
    except Exception:
        return {"name": "", "roll": "", "done": {}, "queue": []}


def _save(s):
    try:
        with open(STORE, "w") as f:
            json.dump(s, f)
    except Exception:
        pass


def _who(s):
    if s["roll"]:
        return
    if os.environ.get("MML_TRACKING") == "off":
        s["name"], s["roll"] = "anonymous", "-"
        return
    try:
        s["name"] = input("Your name (or 'anonymous'): ").strip() or "anonymous"
        s["roll"] = input("Roll number: ").strip() or "-"
    except Exception:
        s["name"], s["roll"] = "anonymous", "-"


def _sending(s):
    return TRACKING and s["name"].lower() != "anonymous" and os.environ.get("MML_TRACKING") != "off"


def _post(url, ev):
    body = json.dumps(dict(ev, token=TOKEN)).encode()
    urllib.request.urlopen(urllib.request.Request(url, body, {"Content-Type": "text/plain"}), timeout=2).read()


def sync():
    """Send queued exercises to the instructor. Silent if offline; they stay queued."""
    s = _load()
    if _sending(s):
        dead = set()
        for ev in s["queue"]:
            for u in SINKS:
                if u in dead or u in ev["sent"]:
                    continue
                try:
                    _post(u, ev)
                    ev["sent"].append(u)
                except Exception:
                    dead.add(u)
        s["queue"] = [e for e in s["queue"] if any(u not in e["sent"] for u in SINKS)]
    _save(s)
    return s


def progress_code():
    """Offline? Paste this code to the instructor (chat or USB) instead."""
    s = _load()
    print(base64.b64encode(json.dumps({"name": s["name"], "roll": s["roll"], "done": s["done"]}).encode()).decode())


def _status(s, n):
    ids = MISSIONS[str(n)]["checkpoints"]
    got = sum(1 for c in ids if c in s["done"])
    if got == len(ids):
        return "done", got, len(ids)
    if n == MISSION or got:
        return "now", got, len(ids)
    return "todo", got, len(ids)


def _html(s):
    cols = {"done": "#2e9e5b", "now": "#3b82f6", "todo": "#8a8f98"}
    parts, stats = [], [_status(s, n) for n in range(len(MISSIONS))]
    for n, (st, got, tot) in enumerate(stats):
        x = 45 + n * 108
        title = MISSIONS[str(n)]["title"]
        if n:
            ok = stats[n - 1][0] == "done"
            parts.append(f'<line x1="{x - 108 + 20}" y1="40" x2="{x - 20}" y2="40" stroke="{cols["done"] if ok else cols["todo"]}" stroke-width="3" stroke-dasharray="{"0" if ok else "5 4"}"/>')
        if st == "done":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="{cols[st]}"/><path d="M{x - 8} 40 l6 7 l11 -14" stroke="#fff" stroke-width="3.5" fill="none" stroke-linecap="round" stroke-linejoin="round"/>')
        elif st == "now":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="4"><animate attributeName="r" values="18;22;18" dur="1.8s" repeatCount="indefinite"/></circle><text x="{x}" y="45" text-anchor="middle" font-size="14" font-weight="700" fill="{cols[st]}">{got}/{tot}</text>')
        else:
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="2.5" stroke-dasharray="4 3"/><text x="{x}" y="45" text-anchor="middle" font-size="14" fill="{cols[st]}">{n}</text>')
        parts.append(f'<text x="{x}" y="82" text-anchor="middle" font-size="12" fill="currentColor" opacity="{1 if st != "todo" else 0.55}">M{n} {title}</text>')
    finished = [n for n, t in enumerate(stats) if t[0] == "done"]
    todo = [f"M{n}" for n, t in enumerate(stats) if t[0] == "todo"]
    line = f"<b>{len(finished)} of {len(stats)} missions finished</b> &middot; now in <b>Mission {MISSION}</b> ({stats[MISSION][1]}/{stats[MISSION][2]} exercises)"
    line += f" &middot; not visited yet: {', '.join(todo)}" if todo else " &middot; every mission visited"
    who = f"{s['name']} ({s['roll']})" if s["roll"] else ""
    return (f'<div style="font-family:sans-serif;max-width:700px"><svg viewBox="0 0 640 96" width="100%" role="img" aria-label="Progress map">{"".join(parts)}</svg>'
            f'<div style="font-size:14px">{line}</div><div style="font-size:12px;opacity:.6">{who}</div></div>')


_shown = []


def show_tracker():
    html = HTML(_html(_load()))
    if _shown:
        try:
            update_display(html, display_id="mml-tracker")
            return
        except Exception:
            pass
    display(html, display_id="mml-tracker")
    _shown.append(1)


def checkpoint(cid):
    """Call after an exercise passes: saves it, tells the instructor (if online), refreshes the map."""
    s = _load()
    _who(s)
    if cid not in s["done"]:
        s["done"][cid] = time.strftime("%H:%M:%S")
        s["queue"].append({"name": s["name"], "roll": s["roll"], "mission": MISSION, "checkpoint": cid, "ts": s["done"][cid], "sent": []})
    _save(s)
    sync()
    show_tracker()


def _start():
    s = _load()
    _who(s)
    if CLOUD_URL and _sending(s):  # Colab gives every notebook a fresh disk, so ask what you already finished elsewhere
        try:
            q = urllib.parse.urlencode({"roll": s["roll"], "token": TOKEN})
            for c in json.load(urllib.request.urlopen(CLOUD_URL + "?" + q, timeout=2)).get("done", []):
                s["done"].setdefault(c, "earlier")
        except Exception:
            pass
    _save(s)
    show_tracker()


_start()

## 1. Is your computer ready?

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

print("Python", sys.version.split()[0])
print("numpy", np.__version__)
print("If you see this, everything works ✔")
checkpoint("m00-ready")

## 2. Meet the data
Each handwritten digit is an $8\times 8$ grid of numbers (0 = white, 16 = black). To a computer, a picture is just a matrix, and you have studied matrices for three years.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target           # X: each row is one picture flattened to 64 numbers
print("pictures:", X.shape[0], " numbers per picture:", X.shape[1])

i = 0                                         # TODO (optional): try another number, e.g. 42
print("label:", y[i])
print(digits.images[i].astype(int))
plt.imshow(digits.images[i], cmap="gray_r"); plt.axis("off"); plt.show()

## 3. Teach a machine to read
We split the pictures: a training set to learn from and a test set it has never seen. The model is logistic regression, which you studied in *ML-I Unit III*. In Mission 2 you will build it yourself.

In [ ]:
np.random.seed(0)
order = np.random.permutation(len(X))
train, test = order[:1300], order[1300:]

model = LogisticRegression(max_iter=2000)
model.fit(X[train], y[train])
print("accuracy on unseen digits:", round(model.score(X[test], y[test]), 3))

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, idx in zip(axes.ravel(), test[:12]):
    ax.imshow(digits.images[idx], cmap="gray_r"); ax.axis("off")
    ax.set_title(f"pred {model.predict(X[idx:idx+1])[0]} (true {y[idx]})")
plt.tight_layout(); plt.show()

## 4. Play: scroll through the test set

In [ ]:
try:
    from ipywidgets import interact, IntSlider
    def look(k=0):
        idx = test[k]
        plt.imshow(digits.images[idx], cmap="gray_r"); plt.axis("off")
        plt.title(f"model says {model.predict(X[idx:idx+1])[0]}, truth {y[idx]}"); plt.show()
    interact(look, k=IntSlider(min=0, max=len(test) - 1, value=0))
except Exception as e:
    print("Widgets unavailable, that is fine:", e)

In [ ]:
checkpoint("m00-done")                      # Mission 0 complete: watch your map update at the top

## You just used...
- a dataset as a **matrix** $X\in\mathbb R^{n\times 64}$
- **logistic regression** (ML-I Unit III) trained on one set and tested on another
- the idea of **generalisation**: accuracy on data the model has not seen

## Your CV line
> *Trained and evaluated a handwritten-digit classifier in Python (scikit-learn) with a train/test split.*

Next: **Mission 1**, where we learn how models learn.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/01_gradient_descent_golf.ipynb)